# Отбор клиентов Excel: Санкт-Петербургский РФ (апр–май–июнь 2026)

Бизнес-логика:
1. Клиент (`ID договора` / `agr_id`) есть во **всех трёх** месяцах.
2. По **5 клиентов** на каждый тарифный сегмент: `0`, `Меню возможностей...`, `По Акту индивидуальный`, `Стандарт`.
3. Поля результата: ID договора, Наименование, ИНН, Номер договора, Дата регистрации договора, Тариф, Сумма операций, Комиссия (% с операций), Комиссия (₽ в месяц).
4. **Стандарт**: дата регистрации ≤ 2026-01-01, нет даты закрытия, сумма операций ≤ 400 000, комиссия ₽/мес = 0.
5. **Меню возможностей**: дата регистрации ≤ 2026-01-01, нет даты закрытия, сумма операций ≤ 0, комиссия ₽/мес = 0.
6. **0** и **По Акту индивидуальный**: TOP-5 по наибольшей `Комиссия (₽ в месяц)` (ранг = max по 3 месяцам, tie-break = sum).

Фильтр филиала: значение содержит `Санкт-Петербургский` (колонка `Филиал` / `Региональный филиал`).

Финальный Excel: **3 листа** — по одному на месяц (`2026-04`, `2026-05`, `2026-06`); на каждом листе одни и те же отобранные `agr_id` с метриками этого месяца.

> В Excel колонка называется `Комиссия (% с операций)` (не «Количество»). В выгрузке сохраняем имя из отчёта.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

In [ ]:
DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
OUT_DIR = DATA_DIR / 'spb_rf_clients_apr_jun_selection'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RF = 'Санкт-Петербургский'
REG_DATE_MAX = pd.Timestamp('2026-01-01')
SAMPLE_PER_TARIFF = 5
STANDARD_TRX_SUM_MAX = 400_000.0
MENU_TRX_SUM_MAX = 0.0
COMM_MONTHLY_EPS = 0.005  # допуск к «0,00»

excel_sources = [
    {'report_month': '2026-04-01', 'path': DATA_DIR / '04_Апрель_2026.xlsx', 'header': 0},
    {'report_month': '2026-05-01', 'path': DATA_DIR / '05_Май_2026.xlsx', 'header': 0},
    {'report_month': '2026-06-01', 'path': DATA_DIR / '06_Июнь_2026.xlsx', 'header': 0},
]
MONTHS = ['2026-04', '2026-05', '2026-06']
TARIFF_ORDER = ['0', 'Меню возможностей', 'По Акту индивидуальный', 'Стандарт']

for src in excel_sources:
    p = Path(src['path'])
    print(f"{src['report_month'][:7]}: exists={p.exists()} | header={src['header']} | {p}")

## 1) Helpers

In [ ]:
def normalize_colname(value):
    s = str(value).lower().replace('\n', ' ').replace('\r', ' ').replace('\xa0', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    s = s.replace('₽', 'руб').replace('%', 'pct')
    s = re.sub(r'[^a-zа-я0-9]+', '', s)
    return s


def pick_column(columns, aliases):
    cols = list(columns)
    norm_map = {normalize_colname(c): c for c in cols}
    for alias in aliases:
        if alias in cols:
            return alias
        key = normalize_colname(alias)
        if key in norm_map:
            return norm_map[key]
    # частичное совпадение по нормализованному имени
    for alias in aliases:
        key = normalize_colname(alias)
        for nk, original in norm_map.items():
            if key and key in nk:
                return original
    return None


def to_num(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace('\xa0', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace(',', '.', regex=False),
        errors='coerce',
    )


def normalize_agr_id(value):
    if pd.isna(value):
        return None
    s = str(value).strip().replace('\xa0', '').replace(' ', '')
    if s.lower() in {'', 'nan', 'none'}:
        return None
    s = re.sub(r'\.0$', '', s)
    return s


def normalize_inn(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else s


def is_empty_date(series):
    """True = даты закрытия нет."""
    s = series.copy()
    as_dt = pd.to_datetime(s, errors='coerce')
    as_str = s.astype(str).str.strip().str.lower()
    empty_str = as_str.isin({'', 'nan', 'none', 'nat', 'na', '<null>', 'null', '-'})
    return as_dt.isna() & (s.isna() | empty_str)


def classify_tariff(value):
    if pd.isna(value):
        return None
    raw = str(value).strip()
    if raw == '' or raw.lower() in {'nan', 'none'}:
        return None

    # тариф «0» как в Excel
    if re.fullmatch(r'0(\.0+)?', raw):
        return '0'

    t = raw.lower().replace('ё', 'е')
    t = re.sub(r'\s+', ' ', t).strip()

    # приоритет: По Акту индивидуальный -> Меню -> Стандарт -> 0 (уже выше)
    has_akt = 'акт' in t
    has_ind = 'индивид' in t
    if has_akt and has_ind:
        return 'По Акту индивидуальный'
    if has_akt and ('по акту' in t or t.startswith('по акт')):
        return 'По Акту индивидуальный'
    if 'меню' in t or 'возможност' in t:
        return 'Меню возможностей'
    if 'стандарт' in t:
        return 'Стандарт'
    return None


COL_ALIASES = {
    'agr_id': ['ID договора', 'agr_id', 'abs_agr_id'],
    'company_name': ['Наименование', 'Наименование клиента', 'Наименование эквайринга', 'company_name'],
    'inn': ['ИНН', 'ИНН клиента', 'inn', 'c_inn'],
    'contract_number': ['Номер договора', 'Номер договора эквайринга', 'n_agr', 'contract_number'],
    'd_valid_from': ['Дата регистрации договора', 'Дата начала договора', 'd_valid_from', 'Дата начала'],
    'd_valid_to': ['Дата закрытия договора', 'Дата окончания договора', 'd_valid_to', 'Дата окончания'],
    'tariff': ['Тариф', 'Тарифный план', 'Тариф клиента', 'Название тарифа', 'Наименование тарифа', 'tariff_name'],
    'trx_sum': ['Сумма операций', 'Сумма опреаций', 'trx_sum'],
    'commission_from_ops': [
        'Комиссия (% с операций)',
        'Комиссия \n(% с операций)',
        'Комиссия % с операций',
        'Количество (% с операций)',  # на случай опечатки в шапке
        'Комиссия эквайринга',
    ],
    'commission_monthly': [
        'Комиссия (₽ в месяц)',
        'Комиссия \n(₽ в месяц)',
        'Комиссия CN (₽ в месяц)',
        'Комиссия в месяц',
        'Комиссия (руб в месяц)',
        'Комиссия (Р в месяц)',
    ],
    'filial': ['Филиал', 'Региональный филиал', 'Филиал договора', 'branch_nm', 'filial_rf'],
}

## 2) Загрузка Excel апр–май–июнь + фильтр РФ

In [ ]:
frames = []
resolved_by_month = []

for src in excel_sources:
    path = Path(src['path'])
    if not path.exists():
        raise FileNotFoundError(f'Нет файла: {path}')

    raw = pd.read_excel(path, header=src['header'])
    resolved = {k: pick_column(raw.columns, aliases) for k, aliases in COL_ALIASES.items()}
    required = ['agr_id', 'tariff', 'trx_sum', 'commission_monthly', 'filial', 'd_valid_from']
    missing = [k for k in required if resolved[k] is None]
    if missing:
        raise ValueError(
            f"{path.name}: не найдены колонки {missing}. "
            f"Доступные: {list(raw.columns)[:40]} ..."
        )

    df = pd.DataFrame({
        'agr_id': raw[resolved['agr_id']].map(normalize_agr_id),
        'company_name': raw[resolved['company_name']] if resolved['company_name'] else None,
        'inn': raw[resolved['inn']].map(normalize_inn) if resolved['inn'] else None,
        'contract_number': raw[resolved['contract_number']] if resolved['contract_number'] else None,
        'd_valid_from': pd.to_datetime(raw[resolved['d_valid_from']], errors='coerce'),
        'd_valid_to': raw[resolved['d_valid_to']] if resolved['d_valid_to'] else np.nan,
        'tariff_raw': raw[resolved['tariff']],
        'trx_sum': to_num(raw[resolved['trx_sum']]),
        'commission_from_ops': to_num(raw[resolved['commission_from_ops']]) if resolved['commission_from_ops'] else np.nan,
        'commission_monthly': to_num(raw[resolved['commission_monthly']]),
        'filial_raw': raw[resolved['filial']],
    })
    df['report_month'] = pd.to_datetime(src['report_month'])
    df['report_month_str'] = df['report_month'].dt.strftime('%Y-%m')
    df['tariff_group'] = df['tariff_raw'].map(classify_tariff)
    df['no_close_date'] = is_empty_date(df['d_valid_to'])
    df['filial_norm'] = (
        df['filial_raw'].astype(str)
        .str.replace('\xa0', ' ', regex=False)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

    # РФ: обрезаем до первого «РФ» включительно (как в витрине) и фильтруем СПб
    df['filial_rf'] = df['filial_norm'].map(
        lambda v: re.match(r'^(.*?РФ)', v).group(1) if isinstance(v, str) and 'РФ' in v else v
    )
    mask_rf = df['filial_norm'].str.contains(TARGET_RF, case=False, na=False)
    df = df[mask_rf & df['agr_id'].notna()].copy()

    frames.append(df)
    resolved_by_month.append({'month': src['report_month'][:7], **resolved, 'rows_spb': len(df), 'rows_raw': len(raw)})
    print(f"{src['report_month'][:7]}: raw={len(raw):,} | SPB={len(df):,} | tariff mapped={df['tariff_group'].notna().sum():,}")

excel_spb = pd.concat(frames, ignore_index=True)
resolved_df = pd.DataFrame(resolved_by_month)
display(resolved_df)
print('Total SPB rows:', len(excel_spb))
print('Tariff groups:')
display(excel_spb['tariff_group'].value_counts(dropna=False))

## 3) Клиенты во всех трёх месяцах + стабильный тарифный сегмент

In [ ]:
# agr_id присутствует в каждом из 3 месяцев
months_per_agr = (
    excel_spb.groupby('agr_id')['report_month_str']
    .nunique()
    .rename('months_cnt')
    .reset_index()
)
stable_ids = set(months_per_agr.loc[months_per_agr['months_cnt'] == len(MONTHS), 'agr_id'])
stable_df = excel_spb[excel_spb['agr_id'].isin(stable_ids)].copy()

# один сегмент тарифа во всех месяцах (не смешиваем переходы тарифа)
group_nunique = stable_df.groupby('agr_id')['tariff_group'].nunique(dropna=True)
stable_tariff_ids = set(group_nunique[group_nunique == 1].index)
# и сегмент не None
has_group = (
    stable_df[stable_df['agr_id'].isin(stable_tariff_ids)]
    .groupby('agr_id')['tariff_group']
    .first()
)
has_group = has_group[has_group.isin(TARIFF_ORDER)]
candidate_df = stable_df[stable_df['agr_id'].isin(set(has_group.index))].copy()
candidate_df['tariff_group'] = candidate_df['agr_id'].map(has_group)

print(f'Уникальных agr_id в SPB: {excel_spb["agr_id"].nunique():,}')
print(f'Во всех {len(MONTHS)} месяцах: {len(stable_ids):,}')
print(f'С одним целевым tariff_group во всех месяцах: {candidate_df["agr_id"].nunique():,}')
display(
    candidate_df.drop_duplicates('agr_id')
    .groupby('tariff_group', as_index=False)['agr_id'].nunique()
    .rename(columns={'agr_id': 'agr_id_cnt'})
)

## 4) Фильтры по сегментам и выбор по 5 клиентов

In [ ]:
def agr_passes_all_months(g, predicate):
    """predicate(row_series) -> bool; должен выполняться в каждом месяце."""
    if g['report_month_str'].nunique() < len(MONTHS):
        return False
    return bool(g.apply(predicate, axis=1).all())


def pred_standard(row):
    return (
        pd.notna(row['d_valid_from']) and row['d_valid_from'] <= REG_DATE_MAX
        and bool(row['no_close_date'])
        and pd.notna(row['trx_sum']) and float(row['trx_sum']) <= STANDARD_TRX_SUM_MAX
        and pd.notna(row['commission_monthly']) and abs(float(row['commission_monthly'])) <= COMM_MONTHLY_EPS
    )


def pred_menu(row):
    return (
        pd.notna(row['d_valid_from']) and row['d_valid_from'] <= REG_DATE_MAX
        and bool(row['no_close_date'])
        and pd.notna(row['trx_sum']) and float(row['trx_sum']) <= MENU_TRX_SUM_MAX
        and pd.notna(row['commission_monthly']) and abs(float(row['commission_monthly'])) <= COMM_MONTHLY_EPS
    )


def build_agr_metrics(df):
    agg = df.groupby('agr_id').agg(
        tariff_group=('tariff_group', 'first'),
        tariff_raw_example=('tariff_raw', lambda s: next((x for x in s if pd.notna(x) and str(x).strip()), None)),
        company_name=('company_name', 'first'),
        inn=('inn', 'first'),
        contract_number=('contract_number', 'first'),
        d_valid_from=('d_valid_from', 'min'),
        commission_monthly_max=('commission_monthly', 'max'),
        commission_monthly_sum=('commission_monthly', 'sum'),
        trx_sum_max=('trx_sum', 'max'),
        trx_sum_sum=('trx_sum', 'sum'),
        months_cnt=('report_month_str', 'nunique'),
    ).reset_index()
    return agg


selected_ids = []
selection_log = []

for tariff in TARIFF_ORDER:
    pool = candidate_df[candidate_df['tariff_group'] == tariff].copy()
    if pool.empty:
        selection_log.append({'tariff_group': tariff, 'candidates': 0, 'selected': 0, 'note': 'пустой пул'})
        continue

    if tariff == 'Стандарт':
        ok_ids = [
            agr for agr, g in pool.groupby('agr_id')
            if agr_passes_all_months(g, pred_standard)
        ]
        metrics = build_agr_metrics(pool[pool['agr_id'].isin(ok_ids)])
        # среди подходящих — стабильные с меньшей суммой операций (не обязательный критерий, для детерминизма)
        metrics = metrics.sort_values(['trx_sum_max', 'agr_id'], ascending=[True, True])
    elif tariff == 'Меню возможностей':
        ok_ids = [
            agr for agr, g in pool.groupby('agr_id')
            if agr_passes_all_months(g, pred_menu)
        ]
        metrics = build_agr_metrics(pool[pool['agr_id'].isin(ok_ids)])
        metrics = metrics.sort_values(['agr_id'], ascending=[True])
    else:
        # 0 и По Акту индивидуальный — без доп. фильтров, TOP по комиссии ₽/мес
        metrics = build_agr_metrics(pool)
        metrics = metrics.sort_values(
            ['commission_monthly_max', 'commission_monthly_sum', 'agr_id'],
            ascending=[False, False, True],
        )

    top = metrics.head(SAMPLE_PER_TARIFF).copy()
    selected_ids.extend(top['agr_id'].tolist())
    selection_log.append({
        'tariff_group': tariff,
        'candidates': int(metrics['agr_id'].nunique()) if len(metrics) else 0,
        'selected': int(len(top)),
        'note': ', '.join(top['agr_id'].astype(str).tolist()) if len(top) else 'нет кандидатов',
    })
    print(f'=== {tariff}: candidates={selection_log[-1]["candidates"]} selected={selection_log[-1]["selected"]} ===')
    if len(top):
        display(top)

selection_log_df = pd.DataFrame(selection_log)
display(selection_log_df)

if not selected_ids:
    raise RuntimeError('Не удалось отобрать ни одного клиента — проверьте фильтры / наличие тарифов в Excel.')

## 5) Итоговая таблица: 3 листа (месяц = лист), одни и те же agr_id

In [ ]:
detail = candidate_df[candidate_df['agr_id'].isin(selected_ids)].copy()

SHOW_COLS = [
    'Сегмент тарифа',
    'ID договора',
    'Наименование',
    'ИНН',
    'Номер договора',
    'Дата регистрации договора',
    'Тариф',
    'Сумма операций',
    'Комиссия (% с операций)',
    'Комиссия (₽ в месяц)',
]

result_cols_map = {
    'agr_id': 'ID договора',
    'company_name': 'Наименование',
    'inn': 'ИНН',
    'contract_number': 'Номер договора',
    'd_valid_from': 'Дата регистрации договора',
    'tariff_raw': 'Тариф',
    'trx_sum': 'Сумма операций',
    'commission_from_ops': 'Комиссия (% с операций)',
    'commission_monthly': 'Комиссия (₽ в месяц)',
}

result_df = detail[[
    'report_month_str', 'tariff_group', 'filial_rf',
    'agr_id', 'company_name', 'inn', 'contract_number',
    'd_valid_from', 'd_valid_to', 'tariff_raw',
    'trx_sum', 'commission_from_ops', 'commission_monthly',
]].copy()

result_df['tariff_group'] = pd.Categorical(result_df['tariff_group'], categories=TARIFF_ORDER, ordered=True)
# стабильный порядок agr_id: по сегменту, затем как в selected_ids
agr_order = {agr: i for i, agr in enumerate(selected_ids)}
result_df['_agr_order'] = result_df['agr_id'].map(agr_order)
result_df = result_df.sort_values(['tariff_group', '_agr_order', 'report_month_str']).reset_index(drop=True)

SHEET_NAME_BY_MONTH = {
    '2026-04': '2026-04',
    '2026-05': '2026-05',
    '2026-06': '2026-06',
}

sheets_by_month = {}
for month in MONTHS:
    part = result_df[result_df['report_month_str'] == month].copy()
    show = part.rename(columns={
        'tariff_group': 'Сегмент тарифа',
        **result_cols_map,
    })
    # на всякий случай — те же agr_id и тот же порядок на каждом листе
    show = show.sort_values(['Сегмент тарифа', '_agr_order']).reset_index(drop=True)
    sheets_by_month[month] = show[SHOW_COLS].copy()

print(f'Selected agr_id: {result_df["agr_id"].nunique()} | rows total: {len(result_df)}')
for month, sdf in sheets_by_month.items():
    print(f'--- sheet {SHEET_NAME_BY_MONTH[month]}: rows={len(sdf)}, agr_id={sdf["ID договора"].nunique()} ---')
    display(sdf)

## 6) Контроль: одни и те же agr_id на всех трёх листах

In [ ]:
ids_by_month = {
    month: set(sdf['ID договора'].astype(str))
    for month, sdf in sheets_by_month.items()
}
base_ids = ids_by_month[MONTHS[0]]
all_equal = all(ids_by_month[m] == base_ids for m in MONTHS)
missing = {
    m: sorted(base_ids - ids_by_month[m])
    for m in MONTHS
    if ids_by_month[m] != base_ids
}

print('Same agr_id set on all 3 sheets:', all_equal)
if not all_equal:
    print('Differences:', missing)
    raise RuntimeError('На листах разные наборы agr_id — проверьте отбор/джойн по месяцам.')

qc_df = (
    result_df.groupby(['tariff_group', 'agr_id'], as_index=False)
    .agg(
        months_cnt=('report_month_str', 'nunique'),
        commission_monthly_max=('commission_monthly', 'max'),
        commission_monthly_sum=('commission_monthly', 'sum'),
        trx_sum_max=('trx_sum', 'max'),
    )
    .sort_values(['tariff_group', 'agr_id'])
)
display(qc_df)
display(selection_log_df)

## 7) Сводка: agr_id × Тариф × апрель / май / июнь

Одна строка на отобранный `agr_id`. В колонках месяцев — **Комиссия (₽ в месяц)** за соответствующий месяц.

In [ ]:
MONTH_COL_RU = {
    '2026-04': 'апрель',
    '2026-05': 'май',
    '2026-06': 'июнь',
}

pivot_src = result_df.copy()
pivot_src['month_ru'] = pivot_src['report_month_str'].map(MONTH_COL_RU)

# если в месяце несколько строк на agr_id — суммируем комиссию
pivot_num = (
    pivot_src.groupby(['agr_id', 'month_ru'], as_index=False)['commission_monthly']
    .sum()
)

wide = pivot_num.pivot(index='agr_id', columns='month_ru', values='commission_monthly')
wide = wide.reindex(columns=['апрель', 'май', 'июнь'])

tariff_map = (
    pivot_src.sort_values(['agr_id', 'report_month_str'])
    .groupby('agr_id', as_index=True)['tariff_raw']
    .first()
)
segment_map = (
    pivot_src.groupby('agr_id', as_index=True)['tariff_group']
    .first()
)

summary_5col = (
    wide.reset_index()
    .assign(Тариф=lambda d: d['agr_id'].map(tariff_map))
    [['agr_id', 'Тариф', 'апрель', 'май', 'июнь']]
)

# порядок как в отборе (по сегментам / selected_ids)
summary_5col['_ord'] = summary_5col['agr_id'].map(agr_order)
summary_5col['_seg'] = summary_5col['agr_id'].map(segment_map)
summary_5col['_seg'] = pd.Categorical(summary_5col['_seg'], categories=TARIFF_ORDER, ordered=True)
summary_5col = (
    summary_5col
    .sort_values(['_seg', '_ord'])
    .drop(columns=['_ord', '_seg'])
    .reset_index(drop=True)
)

assert list(summary_5col.columns) == ['agr_id', 'Тариф', 'апрель', 'май', 'июнь']
assert summary_5col['agr_id'].nunique() == len(selected_ids)
assert len(summary_5col) == len(selected_ids)

print(f'summary_5col: {len(summary_5col)} rows × {len(summary_5col.columns)} cols')
display(summary_5col)

## 8) Выгрузка в Excel

- основной файл: 3 листа по месяцам + `selection_log` + `qc_selected_agr` + `summary_5col`
- отдельный файл только со сводкой 5 колонок

In [ ]:
out_xlsx = OUT_DIR / 'spb_rf_selected_clients_apr_may_jun_2026.xlsx'
out_summary_xlsx = OUT_DIR / 'spb_rf_selected_agr_id_tariff_months.xlsx'
out_summary_csv = OUT_DIR / 'spb_rf_selected_agr_id_tariff_months.csv'

with pd.ExcelWriter(out_xlsx, engine='openpyxl') as writer:
    for month in MONTHS:
        sheet_name = SHEET_NAME_BY_MONTH[month]
        sheets_by_month[month].to_excel(writer, sheet_name=sheet_name, index=False)
    summary_5col.to_excel(writer, sheet_name='summary_5col', index=False)
    selection_log_df.to_excel(writer, sheet_name='selection_log', index=False)
    qc_df.to_excel(writer, sheet_name='qc_selected_agr', index=False)

with pd.ExcelWriter(out_summary_xlsx, engine='openpyxl') as writer:
    summary_5col.to_excel(writer, sheet_name='agr_id_tariff_months', index=False)

summary_5col.to_csv(out_summary_csv, index=False, encoding='utf-8-sig')

print('Saved:', out_xlsx)
print('Sheets:', [SHEET_NAME_BY_MONTH[m] for m in MONTHS] + ['summary_5col', 'selection_log', 'qc_selected_agr'])
for month in MONTHS:
    sdf = sheets_by_month[month]
    n_agr = sdf['ID договора'].nunique()
    print(f'  {SHEET_NAME_BY_MONTH[month]}: {len(sdf)} rows / {n_agr} agr_id')

print('Saved summary-only:')
print(' ', out_summary_xlsx)
print(' ', out_summary_csv)
print(summary_5col.head(10))